16e. Hiệu chỉnh ngưỡng cho baseline và GNN đơn trên NF-ToN-IoT-v3, CPU (task 5.8, GĐ G)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Đường dẫn đầu vào

In [ ]:
DS = "nf_ton_iot_v3"
DATA = os.path.dirname(find(f"{DS}/split.json"))                         # output của notebook 04
CKPT = os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt"))                    # output của 07
OLD = f"{REPO}/reports/kaggle"                                             # kết quả GĐ A, D đã kéo về repo (bản repo, xem 15c)
INFO.update(dataset=DS, data=DATA, ckpt=CKPT)
save_info()
print(DATA, CKPT, sep="\n")

## Ngưỡng cho mọi baseline và GNN đơn trên NF-ToN-IoT-v3
GNN: nạp checkpoint của 07. Baseline: huấn luyện lại với cấu hình tốt nhất của GĐ D (bản `baseline_search_nf_ton_iot_v3.json` trong repo,
khớp kết quả đã lưu theo notebook 15c); F1-macro dựng lại được đối chiếu với `results_baselines_nf_ton_iot_v3.csv` của repo.

In [ ]:
with step("thresholds_baselines_gnn"):
    run([sys.executable, "-I", "scripts/run_thresholds.py", DATA, DS, f"{WORK}/thresholds_base_gnn_{DS}.json",
         f"baseline_search={OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
         f"baseline_csv={OLD}/gnn-iot-ids-05-baselines/results_baselines_{DS}.csv",
         f"gnn_csv={OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv", f"gnn_ckpt={CKPT}"], cwd=REPO)

In [ ]:
thr = json.load(open(f"{WORK}/thresholds_base_gnn_{DS}.json", encoding="utf-8"))[DS]
INFO["thresholds_models"] = {m: sorted(v) for m, v in thr.items()}
INFO["repro_f1_diff_max_abs"] = {m: max(abs(c["repro_f1_diff"]) for c in v.values() if c["repro_f1_diff"] is not None) for m, v in thr.items()}
save_info()
print(json.dumps(INFO["repro_f1_diff_max_abs"], indent=1))
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())